# Create Dave config

Builds the Dave experiment recipe XML from `round_info.csv` (notebook 03) and the Kilroy config for `MICROSCOPE`.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent.parent  # MERci/ (notebook lives in MERci/notebooks/prepare_imaging/<variant>/<acquisition>/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.acquisition.dave    import create_dave_config, dave_config_filename, create_focus_test_dave_config
from MERci.acquisition.kilroy  import find_kilroy_config
from MERci.acquisition.configs import (
    get_frame_table, create_shutter_file, create_hal_config, power_dict_to_channel_list,
    get_color_sequence_name, hal_config_filename, shutter_filename, frame_table_filename,
    read_hal_exposure_time,
)
from MERci.analysis.stage_z    import read_off_file_if_ready, summarize_focus_lock

In [ ]:
SETTINGS_DIR  = SAMPLE_DIR / "settings"
METADATA_DIR  = SAMPLE_DIR / "metadata"
POSITIONS_DIR = SAMPLE_DIR / "positions"

# SAMPLE_NAME is the TRUE top-level experiment id -- must match what notebooks
# 02/03 used, since the positions files and round_info rows below are named
# with it (see notebook 02's docstring for why this isn't SAMPLE_DIR.name).
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
# POSITIONS_TAG is what every positions_{...}.txt filename this notebook reads/builds
# actually uses -- SAMPLE_NAME alone in the flat layout, or SAMPLE_NAME_IMAGING_DIR in
# a split layout.
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")

In [ ]:
# ── Experiment parameters ──────────────────────────────────────────
MICROSCOPE           = "MF3"   # must match what you set in notebooks 01/03
USE_ADAPTORS         = True    # True = adaptor-based fluidics; False = direct readouts
INCLUDE_FINAL_CLEAVE = False   # True = add a final cleave step after last imaging round
FIRST_HYB_NO_CLEAVE  = False   # True = first hyb (after the cells round) omits the cleave step

# Default round structure: imaging round 1 = cells only; rounds 2..N_HYBS+1 = bits #1..#N.
# The fluidics before the first bits round has no cleave; later fluidics include the cleave.

# ── Read what notebook 03 produced ──────────────────────────────────
round_info_path = METADATA_DIR / "round_info.csv"
if not round_info_path.exists():
    raise FileNotFoundError(f"{round_info_path} not found -- run notebook 03 first.")
round_info = pd.read_csv(round_info_path)

rbc_path = METADATA_DIR / "round_bit_color_map.csv"
if not rbc_path.exists():
    raise FileNotFoundError(f"{rbc_path} not found -- run notebook 03 first.")
N_HYBS = int(pd.read_csv(rbc_path)["round"].max())

MULTI_BOUNDARY = "positions_file" in round_info.columns
print(f"N_HYBS               : {N_HYBS}")
print(f"Multi-boundary layout: {MULTI_BOUNDARY}")
print(f"Use adaptors         : {USE_ADAPTORS}")
print(f"Final cleave         : {INCLUDE_FINAL_CLEAVE}")
print(f"First hyb no cleave  : {FIRST_HYB_NO_CLEAVE}")

In [ ]:
# ── Resolve positions inputs for the recipe ──────────────────────────────
if MULTI_BOUNDARY:
    # Per-segment: each round_info row names its own positions file in POSITIONS_DIR.
    positions_arg     = None
    positions_dir_arg = POSITIONS_DIR
    missing = [f for f in round_info["positions_file"].unique()
               if not (POSITIONS_DIR / f).exists()]
    if missing:
        raise FileNotFoundError(
            f"Positions files referenced by round_info are missing (run notebook 02): {missing}"
        )
else:
    # Single-positions: one file for every movie.
    positions_arg     = POSITIONS_DIR / f"positions_{POSITIONS_TAG}.txt"
    positions_dir_arg = None
    if not positions_arg.exists():
        raise FileNotFoundError(f"Positions file not found: {positions_arg}")

# Resolve the Kilroy config that will run this experiment. Its protocol names are
# the source of truth for the fluidic steps written into the Dave recipe, so every
# protocol referenced is guaranteed to exist in Kilroy. If the microscope has no
# Kilroy config, fall back to MF2's.
KILROY_DIR    = MERCI_DIR / "data" / "configs" / "kilroy"
KILROY_CONFIG = find_kilroy_config(MICROSCOPE, KILROY_DIR, fallback_microscope="MF2")
print(f"Kilroy config (protocol source): {KILROY_CONFIG.name}")

dave_output = SETTINGS_DIR / dave_config_filename(MICROSCOPE, N_HYBS, SAMPLE_NAME)

create_dave_config(
    round_info           = round_info,
    positions_file       = positions_arg,
    settings_dir         = SETTINGS_DIR,
    output_path          = dave_output,
    use_adaptors         = USE_ADAPTORS,
    include_final_cleave = INCLUDE_FINAL_CLEAVE,
    first_hyb_no_cleave  = FIRST_HYB_NO_CLEAVE,
    kilroy_config        = KILROY_CONFIG,
    positions_dir        = positions_dir_arg,
)

print(f"Dave config saved: {dave_output}")

# Preview the generated file
with open(dave_output, encoding="ISO-8859-1") as fh:
    print(fh.read())

## Kilroy config

Locates the Kilroy config for the current microscope in `MERci/data/configs/kilroy/`
(newest matching file by `YYMMDD` date stamp) and copies it to `SAMPLE_DIR/settings/`.
This is the same file used above as the protocol source for the Dave recipe. If the
microscope has no Kilroy config, it falls back to MF2's.

In [7]:
import shutil

# Same resolution (with MF2 fallback) used as the Dave protocol source above.
kilroy_src  = find_kilroy_config(MICROSCOPE, MERCI_DIR / "data" / "configs" / "kilroy",
                                 fallback_microscope="MF2")
kilroy_dest = SETTINGS_DIR / kilroy_src.name
shutil.copy2(str(kilroy_src), str(kilroy_dest))
print(f"Kilroy config  : {kilroy_src.name}")
print(f"Copied to      : {kilroy_dest}")

Kilroy config  : kilroy-config-mf4-direct-and-adaptors-260617.xml
Copied to      : c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\settings\kilroy-config-mf4-direct-and-adaptors-260617.xml


## Focus-lock test recipe (optional)

Builds a **separate** Dave recipe that visits every FOV and checks focus
lock only -- no fluidics -- to catch a bad lock across the whole coverslip
before committing to the full multi-hour acquisition above. Does not touch
`dave_output`; writes its own file.

**`N_TEST_FRAMES = 0` (default): check-focus only, no image ever taken.**
Confirmed directly against the real, unmodified Dave source (`v2Generator`/
`daveActions`, run against a generated recipe of each kind): a `<movie>`
that omits `<length>`/`<parameters>` expands to a branch containing ONLY
`DAMoveStage` + `DACheckFocus` -- no `DASetParameters`/`DATakeMovie`, no
image taken, no HAL settings changed. No patch to Dave/HAL is needed.

Trade-off: this mode leaves **no persisted per-FOV record** of whether the
lock was good. HAL's focus-status reply only reaches Dave live over TCP;
Dave shows a failed check in its own transient, in-memory warnings list
(never written to a file) -- watch that panel while the recipe runs.

**`N_TEST_FRAMES > 0`: also takes a real (short) movie per FOV.** Needs a
real HAL config/shutter pair, now **auto-generated** below (`FOCUS_TEST_*`
parameters) rather than requiring one to already exist in `settings/` --
a short, representative movie (`FOCUS_TEST_COLOR_SEQ`, e.g. `[488, nan]`,
imaged at a single z position) built with the same `get_frame_table`/
`create_shutter_file`/`create_hal_config` primitives notebook 01 uses for
the real bits/cells/transit rounds, reusing the real bits round's own
exposure time (via `read_hal_exposure_time`) so the test reflects actual
imaging conditions. This produces a genuine, persisted per-FOV record:
HAL's normal `.off` sidecar file (the same one `analysis/stage_z.py`
already reads for stage-z drift), whose `good-offset` column flags exactly
which frame(s) had a bad lock -- at the cost of real (small) disk usage/time
per FOV. Read the results back with the last cell below, after running the
recipe on the microscope.

In [ ]:
# ── Focus-lock test parameters ──────────────────────────────────────────
N_TEST_FRAMES    = 0        # 0 = check-focus only (no movie, no persisted per-FOV
                              # record -- see markdown above); >0 = also take this many
                              # real frames per FOV, producing a real .off file per FOV.
NUM_FOCUS_CHECKS = 50
FOCUS_SCAN       = True

# ── Auto-generate the focus-test HAL config + shutter ────────────────────
# A short, representative movie: one real color (488, the bead-focus channel)
# plus a blank companion frame, imaged at a single z position -- just enough
# for HAL to write a real per-FOV .off sidecar, not a full z-stack. Built
# unconditionally (cheap) so it's already there the moment N_TEST_FRAMES is
# bumped above 0, using the same get_frame_table/create_shutter_file/
# create_hal_config primitives notebook 01 uses for the real rounds.
FOCUS_TEST_COLOR_SEQ     = [488, np.nan]   # get_frame_table color_seq notation
FOCUS_TEST_Z             = 0.5             # single z position (um above bead_z)
FOCUS_TEST_POWER         = {488: 1.00}
FOCUS_TEST_POWER_DEFAULT = 1.0

_hal_dir        = MERCI_DIR / "data" / "configs" / "hal"
_hal_candidates = sorted(
    p for p in _hal_dir.glob("hal-config-*.xml")
    if MICROSCOPE.lower() in p.name.lower()
)
if not _hal_candidates:
    raise FileNotFoundError(f"No HAL template found for microscope '{MICROSCOPE}' in {_hal_dir}")
FOCUS_TEST_HAL_TEMPLATE = _hal_candidates[0]

# Reuse the real bits round's own exposure time (same lookup notebook 06 uses
# for experiment_info.yaml) so the test reflects actual imaging conditions,
# rather than guessing a new value; falls back to a sensible default if no
# bits HAL config has been generated yet.
_bits_hal_configs = sorted(SETTINGS_DIR.glob("hal-config-*bits*.xml"))
FOCUS_TEST_EXPOSURE_TIME = (
    read_hal_exposure_time(_bits_hal_configs[-1]) if _bits_hal_configs else 0.15
)

focus_test_frame_table = get_frame_table(
    bead_z=0, bead_seq=[], color_seq=FOCUS_TEST_COLOR_SEQ, end_seq=[],
    z_pos=np.array([FOCUS_TEST_Z]), microscope=MICROSCOPE,
)
focus_test_name = get_color_sequence_name(focus_test_frame_table)

focus_test_ft_path = METADATA_DIR / frame_table_filename("focustest", focus_test_name)
focus_test_frame_table.to_csv(focus_test_ft_path)

focus_test_shutter_path = SETTINGS_DIR / shutter_filename("focustest", focus_test_name)
create_shutter_file(focus_test_frame_table, focus_test_shutter_path,
                     default_power=FOCUS_TEST_POWER_DEFAULT)

TEST_HAL_CONFIG = hal_config_filename(MICROSCOPE, "focustest", focus_test_name)
create_hal_config(
    FOCUS_TEST_HAL_TEMPLATE, focus_test_frame_table, focus_test_shutter_path.name,
    SETTINGS_DIR / TEST_HAL_CONFIG,
    default_power=power_dict_to_channel_list(FOCUS_TEST_POWER, MICROSCOPE, FOCUS_TEST_POWER_DEFAULT),
    file_type=".zarr", exposure_time=FOCUS_TEST_EXPOSURE_TIME,
)
print(f"Focus-test frame table: {focus_test_ft_path.name}  ({len(focus_test_frame_table)} frame(s))")
print(f"Focus-test shutter    : {focus_test_shutter_path.name}")
print(f"Focus-test HAL config : {TEST_HAL_CONFIG}  (exposure {FOCUS_TEST_EXPOSURE_TIME}s)")

# tumor/ is single-tissue-per-coverslip, so notebook 02 always writes this aggregate
# FOV file (regardless of MULTI_BOUNDARY/segment mode above, which only concerns
# multiple BOUNDARIES within that one tissue, not multiple tissues).
focus_test_positions = POSITIONS_DIR / f"positions_{POSITIONS_TAG}.txt"
if not focus_test_positions.exists():
    raise FileNotFoundError(f"{focus_test_positions} not found -- run notebook 02 first.")

focus_test_output   = SETTINGS_DIR / f"dave-{MICROSCOPE.lower()}-focustest-{SAMPLE_NAME}.xml"
focus_test_data_dir = (SAMPLE_DIR / "data" / "focus_test") if N_TEST_FRAMES > 0 else None

n_fovs = create_focus_test_dave_config(
    positions_file   = focus_test_positions,
    output_path      = focus_test_output,
    num_focus_checks = NUM_FOCUS_CHECKS,
    focus_scan       = FOCUS_SCAN,
    n_test_frames    = N_TEST_FRAMES,
    hal_config       = TEST_HAL_CONFIG,
    settings_dir     = SETTINGS_DIR,
    data_dir         = focus_test_data_dir,
    movie_name       = f"hal-{MICROSCOPE.lower()}-focustest",
)

print(f"Focus-lock test recipe saved: {focus_test_output}")
print(f"FOVs visited: {n_fovs}")
if N_TEST_FRAMES > 0:
    print(f"Each FOV takes a real {N_TEST_FRAMES}-frame movie -> .off files land under {focus_test_data_dir}")
else:
    print("Check-focus only -- no movie taken, no persisted per-FOV record (see markdown above).")

with open(focus_test_output, encoding="ISO-8859-1") as fh:
    print(fh.read())

In [ ]:
# ── Read back per-FOV focus-lock results ──────────────────────────────────
# Only meaningful when N_TEST_FRAMES > 0 above AND the recipe has already been
# run on the microscope (each FOV's .off sidecar only exists once HAL has
# actually written that FOV's movie -- see the markdown above).
if N_TEST_FRAMES > 0:
    # Dave's own auto-increment padding (v2Generator's copyChildren) zero-pads
    # the FOV index to len(str(n_fovs)) digits -- confirmed directly against the
    # real source; distinct from MERci's own fov_pad_width convention used for
    # round_info.csv series patterns, so it is NOT reused here.
    pad = len(str(n_fovs))
    bad_fovs  = []
    n_checked = 0
    for fov_idx in range(n_fovs):
        off_path = focus_test_data_dir / f"hal-{MICROSCOPE.lower()}-focustest_{str(fov_idx).zfill(pad)}.off"
        # read_off_file_if_ready returns None both when the file doesn't exist
        # yet AND when HAL has created it but not finished writing it (a real
        # race when reading back while the recipe is still running on the
        # microscope) -- either way, just means "not ready yet, check again later".
        off_df = read_off_file_if_ready(off_path)
        if off_df is None:
            continue
        n_checked += 1
        summary = summarize_focus_lock(off_df)
        if not summary["all_good"]:
            bad_fovs.append((fov_idx, summary))

    print(f"Checked {n_checked} / {n_fovs} FOV(s) (missing ones haven't been imaged yet).")
    if bad_fovs:
        print(f"{len(bad_fovs)} FOV(s) with a bad focus lock:")
        for fov_idx, summary in bad_fovs:
            print(f"  FOV {fov_idx}: {summary['n_bad_frames']}/{summary['n_frames']} bad frame(s)")
    elif n_checked:
        print("All checked FOVs had a good focus lock.")
    else:
        print("No FOVs found yet -- run the recipe on the microscope first.")
else:
    print("N_TEST_FRAMES == 0 -- no per-FOV file was written (see markdown above); "
          "check Dave's own warnings panel while the recipe runs instead.")